# Configuration

Every name the agent notebooks share: the Unity Catalog tables it reads, the secret scope, the
model-role table and the tuning constants. **This notebook only defines.** It builds no client,
opens no connection and reads no table, so any notebook can `%run` it for the cost of assigning
some variables.

## Where a value lives

Non-sensitive configuration is a **literal here**; only real credentials live in the Databricks
secret scope, read with `dbutils.secrets.get`. That is `ARCHITECTURE_V2.md` §2 principle 4 —
*secret names in config, secret values only in the scope* — and it keeps the table names visible
in the repo, so `docs/development/contracts/contract1.md` and this file can be confronted by
reading them side by side.

**There is no `os.environ` anywhere in the agent tree.** Environment variables were only ever
needed to get a secret into the Model Serving container, which has no `dbutils`; how the served
copy gets its configuration is deferred along with the deploy design.

## What is deliberately not here yet

A name arrives in this notebook together with the notebook that consumes it — inventing it
earlier means inventing a value nobody has confirmed:

| Missing | Arrives with | Why not now |
|---|---|---|
| the four model names (`agent-design.md` §4.1) | `llm/gateway` | the router and embeddings services are Unity Catalog model services whose full names this file has not seen |
| Vector Search endpoint and index names | `tools/news_tools` | `contract1.md` §5.3 asks the data engineer to hand both names over; guessing them here would defeat that |
| MLflow experiment path, registered model name, `champion` alias | `serving/*` | no consumer until the agent is packaged |
| semantic cache TTLs | `cache/*` | the cache is designed, not built (`agent-design.md` §14.1) |
| gold-table freshness SLA | `tools/panel_data` | it is that notebook's check that asserts freshness (§19.3), and the SLA is still open in `contract1.md` §9 |
| `fundamentals` table | contract 2 | no upstream source ingests it today |

The 300-token output floor is **not** here either: it is a measured property of the gateway, so
per §3.1 it stays in `llm/gateway` next to the note recording how it was measured.

## How to run it

Run this notebook directly and the last cell executes its own `check` and prints one line per
assertion. A notebook that `%run`s this one sets `FINHIVE_SKIP_CHECK = True` **before** the
`%run`, so config's check does not fire once per caller:

```python
FINHIVE_SKIP_CHECK = True
%run ../setup/config
```

`%run` executes the target inline in the caller's namespace, so `__name__` stays `"__main__"`
and an `if __name__ == "__main__":` guard would fire on every `%run` — which is why the flag
exists instead. Forgetting it costs a redundant check, never a wrong result.


In [ ]:
# ---------------------------------------------------------------------------
# Profiles
# ---------------------------------------------------------------------------
# There is exactly one environment (ARCHITECTURE_V2.md §7). The concept survives with a single
# value so that no code ever special-cases "the only environment" as if it were unconfigured.
KNOWN_PROFILES = ("free",)
DEFAULT_PROFILE = "free"

# ---------------------------------------------------------------------------
# Unity Catalog - what the agent reads
# ---------------------------------------------------------------------------
# The agent reads the gold schema and nothing else. It never touches finhive.yahoo or
# finhive.fred: those are the data engineer's, and their grain is one table per series, which is
# the opposite of what the agent needs (contract1.md §1).
CATALOG = "finhive"
GOLD_SCHEMA = "gold"
GOLD_NAMESPACE = f"{CATALOG}.{GOLD_SCHEMA}"

TBL_INSTRUMENTS = f"{GOLD_NAMESPACE}.instruments"
TBL_TECHNICAL = f"{GOLD_NAMESPACE}.technical_indicators"
TBL_RISK = f"{GOLD_NAMESPACE}.risk_metrics"
TBL_CORRELATIONS = f"{GOLD_NAMESPACE}.correlations"
TBL_MACRO = f"{GOLD_NAMESPACE}.macro_series"
TBL_MACRO_CATALOG = f"{GOLD_NAMESPACE}.macro_series_catalog"
TBL_NEWS = f"{GOLD_NAMESPACE}.news"

ALL_GOLD_TABLES = (
    TBL_INSTRUMENTS,
    TBL_TECHNICAL,
    TBL_RISK,
    TBL_CORRELATIONS,
    TBL_MACRO,
    TBL_MACRO_CATALOG,
    TBL_NEWS,
)

# Slice A of contract1.md: the two tables that unblock the first expert end to end. Kept
# separate so tools/panel_data can load and be checked before the other slices are delivered.
SLICE_A_TABLES = (TBL_INSTRUMENTS, TBL_TECHNICAL)

In [ ]:
# ---------------------------------------------------------------------------
# Secrets - names only, never a value
# ---------------------------------------------------------------------------
SECRET_SCOPE = "finhive"

# The agent needs no credential of its own yet. `fred_api_key`, already in this scope, belongs to
# data_ingestion and is read by its worker - not by anything here. A key is declared below when
# the notebook that reads it is written:
#     news provider key  ->  tools/news_tools      (contract1.md slice D)
#     valkey URI         ->  cache/valkey          (agent-design.md §14.1)
#
# The check asserts every key declared here exists in the scope, so this stays correct as it
# fills up rather than only today, while it is empty.
AGENT_SECRET_KEYS = {}

In [ ]:
# ---------------------------------------------------------------------------
# Model roles - sampling parameters only
# ---------------------------------------------------------------------------
# A role sets the model, the temperature and the token cap; it tags the MLflow trace so a run
# shows which part of the graph spent tokens; and it documents intent (agent-design.md §4.1).
# The role -> model-name map is llm/gateway's, and it is the only code allowed to hold it.
KNOWN_MODEL_ROLES = ("router", "worker", "synthesizer", "guard_in", "guard_out", "embedding")

ROLE_SETTINGS = {
    "router": {"temperature": 0.0, "max_tokens": 600},
    "worker": {"temperature": 0.1, "max_tokens": 1600},
    "synthesizer": {"temperature": 0.2, "max_tokens": 2000},
    "guard_in": {"temperature": 0.0, "max_tokens": 300},
    "guard_out": {"temperature": 0.0, "max_tokens": 800},
    "embedding": {},  # an embedding call takes no sampling parameters
}

# ---------------------------------------------------------------------------
# Tuning constants
# ---------------------------------------------------------------------------
# Measured algorithm thresholds (the consensus 0.5 and 0.35, the MMR lambda, the dedupe 0.62) are
# NOT here: each stays in the file that owns it, next to the note recording how it was measured
# (agent-design.md §3.1). What lives here is what more than one notebook has to agree on.
MAX_SYNTHESIS_ATTEMPTS = 3  # §11.2 - total drafts, not extra retries
DEFAULT_MAX_CONCURRENCY = 4  # §12 - the whole panel; pay-per-token endpoints rate-limit if pushed
GRAPH_RECURSION_LIMIT = 40  # §12 - ample room for 3 drafts
MODEL_TIMEOUT_SECONDS = 120  # first guess; re-measure once the graph runs end to end

## `check`


In [ ]:
def check(ctx):
    """Smoke test for this notebook. Returns a result dict; never raises, never prints a secret.

    Checks what can be checked without opening a table: a gold table's existence and freshness is
    tools/panel_data's check, not this one (agent-design.md §19.3). The one environment fact it
    does assert is that the secret scope is *listable* by whoever is running - the cheapest
    possible proof that this identity can reach secrets at all, which everything later depends on.
    It lists scopes and key names; it never reads a value.
    """
    results = []

    def record(name, ok, detail):
        results.append({"check": name, "ok": bool(ok), "detail": detail})

    # 1. the profile resolves to one this configuration knows
    profile = ctx.get("profile")
    record("profile_known", profile in KNOWN_PROFILES, f"profile={profile!r} known={KNOWN_PROFILES}")

    # 2. every table name is a fully qualified three-part Unity Catalog name
    malformed = [t for t in ALL_GOLD_TABLES if len(t.split(".")) != 3 or not all(t.split("."))]
    record("tables_fully_qualified", not malformed,
           f"malformed={malformed}" if malformed else f"{len(ALL_GOLD_TABLES)} names, all 3-part")

    # 3. no name is declared twice - a duplicate silently makes two variables one table
    duplicated = sorted({t for t in ALL_GOLD_TABLES if ALL_GOLD_TABLES.count(t) > 1})
    record("tables_unique", not duplicated,
           f"duplicated={duplicated}" if duplicated else "no duplicates")

    # 4. every table sits in the gold namespace - catches a typo when a table is added, and a
    #    table accidentally pointed at a source schema the agent must not read
    misplaced = [t for t in ALL_GOLD_TABLES if not t.startswith(f"{GOLD_NAMESPACE}.")]
    record("tables_in_gold_namespace", not misplaced,
           f"misplaced={misplaced}" if misplaced else f"all under {GOLD_NAMESPACE}")

    # 5. slice A is part of the full set, not a third spelling of the same tables
    stray = [t for t in SLICE_A_TABLES if t not in ALL_GOLD_TABLES]
    record("slice_a_subset", not stray, f"not in ALL_GOLD_TABLES={stray}" if stray else "subset")

    # 6. the role table covers exactly the known roles - an unknown role raises in the gateway,
    #    and a missing one fails at the first question instead of here
    missing = [r for r in KNOWN_MODEL_ROLES if r not in ROLE_SETTINGS]
    extra = [r for r in ROLE_SETTINGS if r not in KNOWN_MODEL_ROLES]
    record("roles_complete", not missing and not extra, f"missing={missing} extra={extra}")

    incomplete = [
        r for r in KNOWN_MODEL_ROLES
        if r != "embedding" and not {"temperature", "max_tokens"} <= set(ROLE_SETTINGS.get(r, {}))
    ]
    record("role_settings_complete", not incomplete,
           f"incomplete={incomplete}" if incomplete else "every chat role has temperature + max_tokens")

    # 7. the tuning constants are mutually consistent. The recursion limit has to leave room for
    #    every draft: each retry costs one synthesizer and one output_guardrail superstep on top
    #    of the fixed nodes, so raising MAX_SYNTHESIS_ATTEMPTS without raising the limit would
    #    kill the graph mid-retry rather than ship a warned answer.
    floor = 8 + 2 * MAX_SYNTHESIS_ATTEMPTS
    sane = (
        MAX_SYNTHESIS_ATTEMPTS >= 1
        and DEFAULT_MAX_CONCURRENCY >= 1
        and MODEL_TIMEOUT_SECONDS > 0
        and GRAPH_RECURSION_LIMIT >= floor
    )
    record("tuning_consistent", sane,
           f"attempts={MAX_SYNTHESIS_ATTEMPTS} concurrency={DEFAULT_MAX_CONCURRENCY} "
           f"recursion={GRAPH_RECURSION_LIMIT} (floor {floor}) timeout={MODEL_TIMEOUT_SECONDS}s")

    # 8. the secret scope is reachable, and every declared key exists in it
    handle = ctx.get("dbutils")
    if handle is None:
        record("secret_scope_readable", False, "no dbutils in ctx")
    else:
        try:
            scopes = [s.name for s in handle.secrets.listScopes()]
            present = SECRET_SCOPE in scopes
            record("secret_scope_readable", present,
                   f"scope={SECRET_SCOPE!r} present={present}")
            if present:
                known = {k.key for k in handle.secrets.list(SECRET_SCOPE)}
                absent = sorted(v for v in AGENT_SECRET_KEYS.values() if v not in known)
                record("agent_secret_keys_present", not absent,
                       f"absent={absent}" if absent
                       else f"{len(AGENT_SECRET_KEYS)} declared, all present")
        except Exception as exc:
            record("secret_scope_readable", False, f"{type(exc).__name__}: {exc}")

    return {
        "notebook": "setup/config",
        "profile": profile,
        "ok": all(r["ok"] for r in results),
        "checks": results,
    }

In [ ]:
# Runs this notebook's own check when it is the notebook being executed. A caller that %runs
# config sets FINHIVE_SKIP_CHECK = True first (see the header cell) - %run shares this namespace,
# so an `if __name__ == "__main__"` guard would fire on every caller instead.
# Locals are underscore-prefixed to keep the shared namespace clean (agent-design.md §3.2).
if not globals().get("FINHIVE_SKIP_CHECK", False):
    dbutils.widgets.text("profile", DEFAULT_PROFILE)
    _result = check({"profile": dbutils.widgets.get("profile") or DEFAULT_PROFILE,
                     "dbutils": dbutils})

    for _row in _result["checks"]:
        print(f"{'PASS' if _row['ok'] else 'FAIL'}  {_row['check']:<26}  {_row['detail']}")
    print(f"\nsetup/config [{_result['profile']}]: {'OK' if _result['ok'] else 'FAILED'}")

    if not _result["ok"]:
        raise RuntimeError(
            "setup/config check failed: "
            f"{[_r['check'] for _r in _result['checks'] if not _r['ok']]}"
        )